<a href="https://colab.research.google.com/github/andrewmacc04/SJSU-Work/blob/main/Exercise1_Prompt_Chaining_OFFLINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 – Prompt Chaining for a Customer Support AI (Offline / No API)

**Goal:** Show a multi step customer support flow where the output of one step becomes input to the next.

**Important:** This notebook is **offline**. It does **not** call any API, so you **do not need an API key**.
We simulate the model outputs so the chain logic is still clear and runnable in Colab.

Example customer message used:
> Hi, I was charged twice for my order 38172 and need one charge removed today. I already emailed last week.


In [1]:
# No installs needed for the offline version
import json


In [2]:
# ---------------- Step 0: Input ----------------
# The "customer_message" is what would normally be fed into Step 1.
customer_message = "Hi, I was charged twice for my order 38172 and need one charge removed today. I already emailed last week."
print("Customer message:")
print(customer_message)


Customer message:
Hi, I was charged twice for my order 38172 and need one charge removed today. I already emailed last week.


## Step 1 prompt (Classify issue)

**Prompt used (what I would send to the model):**
- Role: customer support triage assistant
- Task: classify the issue and output JSON with category, urgency, key facts, missing info questions
- Constraint: output valid JSON only


In [8]:
# ---------------- After summary (revised) ----------------
after_summary = (
    "Capitalism is an economic system in which most businesses are privately owned and prices are largely set "
    "through supply and demand. Supporters argue that competition can encourage innovation, efficiency, and "
    "a wider variety of goods and services.\n\n"
    "Critics argue that capitalism can increase economic inequality and allow large firms to gain influence that "
    "reduces real competition. Because outcomes can vary depending on laws and market conditions, many systems "
    "use regulations and safety nets to limit harms while preserving the benefits of competition."
)

print("AFTER SUMMARY:")
print(after_summary)
print("\nWord count:", len(after_summary.split()))

AFTER SUMMARY:
Capitalism is an economic system in which most businesses are privately owned and prices are largely set through supply and demand. Supporters argue that competition can encourage innovation, efficiency, and a wider variety of goods and services.

Critics argue that capitalism can increase economic inequality and allow large firms to gain influence that reduces real competition. Because outcomes can vary depending on laws and market conditions, many systems use regulations and safety nets to limit harms while preserving the benefits of competition.

Word count: 82


## Step 2 prompt (Gather missing info)

**Prompt used:**
- Use Step 1 JSON
- Ask only the missing info questions
- Friendly tone
- Under 80 words
- Do not propose solutions yet


In [4]:
# ---------------- Step 2: Generate the customer questions from Step 1 ----------------
# This is still "offline", but we're demonstrating how Step 2 depends on Step 1 output.

questions = step1_output["missing_info_questions"]

step2_message = (
    "Thanks — I can help with that. Since this is time sensitive, I’ll move quickly. "
    + " ".join([f"{i+1}. {q}" for i, q in enumerate(questions)])
)

# Simple length check (shows constraints)
print("STEP 2 OUTPUT (message to customer):")
print(step2_message)
print("\nWord count:", len(step2_message.split()))


STEP 2 OUTPUT (message to customer):
Thanks — I can help with that. Since this is time sensitive, I’ll move quickly. 1. Can you confirm the last 4 digits of the card used? 2. What email address did you use for the order? 3. What are the dates/amounts of the two charges as they appear on your statement?

Word count: 52


## Step 3 prompt (Propose solution)

**Prompt used:**
- Use Step 1 JSON + customer answers
- Output format:
  1. Summary (1 sentence)
  2. Steps to resolve
  3. What I need from you (only if still missing)
  4. What happens next


In [5]:
# ---------------- Step 3: Pretend customer answered Step 2 ----------------
# In a real conversation, the user would provide these. For the demo we simulate.
customer_answers = {
    "last4": "1234",
    "email": "andrew@example.com",
    "charge_details": "Two charges of $59.99 on Feb 23 and Feb 24"
}
print("Customer answers (simulated):")
print(json.dumps(customer_answers, indent=2))


Customer answers (simulated):
{
  "last4": "1234",
  "email": "andrew@example.com",
  "charge_details": "Two charges of $59.99 on Feb 23 and Feb 24"
}


In [6]:
# ---------------- Step 3: Create final support response using Step 1 + answers ----------------
# This demonstrates chaining: we rely on category, urgency, and facts from Step 1.

summary = "It looks like you were billed twice for the same order, and we can get the duplicate charge reversed."
steps = [
    f"Confirm the order details (Order 38172, email {customer_answers['email']}).",
    f"Verify the duplicate charges ({customer_answers['charge_details']}) tied to card ending {customer_answers['last4']}.",
    "Submit a duplicate charge correction request so the extra charge is reversed.",
    "Email you a confirmation once the reversal is submitted."
]

still_missing = []
if not customer_answers.get("last4"): still_missing.append("Last 4 digits of the card")
if not customer_answers.get("email"): still_missing.append("Email used for the order")
if not customer_answers.get("charge_details"): still_missing.append("Dates/amounts of the charges")

step3_response = []
step3_response.append("1. " + summary)
step3_response.append("2. Steps to resolve:")
for i, s in enumerate(steps, 1):
    step3_response.append(f"   {i}. {s}")
step3_response.append("3. What I need from you: " + ("Nothing else right now." if not still_missing else ", ".join(still_missing)))
step3_response.append("4. What happens next: I’ll confirm once the duplicate charge reversal is submitted.")

print("STEP 3 OUTPUT (solution response):")
print("\n".join(step3_response))


STEP 3 OUTPUT (solution response):
1. It looks like you were billed twice for the same order, and we can get the duplicate charge reversed.
2. Steps to resolve:
   1. Confirm the order details (Order 38172, email andrew@example.com).
   2. Verify the duplicate charges (Two charges of $59.99 on Feb 23 and Feb 24) tied to card ending 1234.
   3. Submit a duplicate charge correction request so the extra charge is reversed.
   4. Email you a confirmation once the reversal is submitted.
3. What I need from you: Nothing else right now.
4. What happens next: I’ll confirm once the duplicate charge reversal is submitted.


## Step 4 prompt (Escalation rule)

**Escalate if any are true:**
1. Urgency is high and customer cannot access account  
2. Payment dispute, chargeback, fraud terms appear  
3. Safety hazard terms appear  
4. Legal threats appear  
5. Still missing required info after asking  

Output JSON: escalate, reason, team


In [7]:
# ---------------- Step 4: Escalation decision based on rules ----------------
text = (customer_message + " " + json.dumps(customer_answers)).lower()

escalate = False
reason = "none"
team = "none"

# Rule 2: payment dispute / chargeback / fraud keywords
dispute_words = ["chargeback", "fraud", "dispute", "stolen", "unauthorized"]
if any(w in text for w in dispute_words):
    escalate = True
    reason = "Possible payment dispute/fraud language detected."
    team = "billing"

# Rule 5: missing required info after asking
required_fields = ["last4", "email", "charge_details"]
if any(not customer_answers.get(k) for k in required_fields):
    escalate = True
    reason = "Missing required billing verification info after requesting it."
    team = "billing"

# Otherwise, for this demo: high urgency billing issue but resolvable
if not escalate and step1_output["category"] == "Billing":
    escalate = False
    reason = "Billing issue can be handled with provided verification details."
    team = "none"

step4_output = {"escalate": escalate, "reason": reason, "team": team}

print("STEP 4 OUTPUT (escalation JSON):")
print(json.dumps(step4_output, indent=2))


STEP 4 OUTPUT (escalation JSON):
{
  "escalate": false,
  "reason": "Billing issue can be handled with provided verification details.",
  "team": "none"
}


## Evidence of testing + iteration (example)

**Run 1 issue:** Step 2 message was too long and included troubleshooting.  
**Fix:** Enforced “ask only questions” and added a word count check.  
**Run 2:** Step 2 stayed under the limit and only asked for missing info.
